# 7. Multi-Modal Integration: Microbiome + Clinical Metadata

**Goal:** In previous notebooks, we optimized predictive models using strictly microbiome data (Genera CLR). In this final step, we test the hypothesis that integrating clinical metadata (Age and Gender) via early integration will enhance predictive performance.

**Methodology:**
We utilize Scikit-Learn's `ColumnTransformer` to create a dual-routing pipeline:
1. Microbiome features undergo their respective dimensionality reduction (PCA for CD, Kruskal-Wallis for UC).
2. Clinical features bypass reduction but undergo standard scaling.
3. Both streams are concatenated and fed into the ultra-optimized non-linear Support Vector Machines (SVM). We keep SVM hyperparameters locked to conduct a rigorous ablation study.

In [11]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.base import BaseEstimator, TransformerMixin
from scipy.stats import kruskal
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.model_selection import RandomizedSearchCV
import warnings
import joblib
warnings.filterwarnings("ignore")

# 1. Load Data
meta = pd.read_csv("../data/processed/metadata_final.tsv", sep="\t", index_col="Sample")
genera_clr = pd.read_csv("../data/processed/genera_clr.tsv", sep="\t", index_col=0)
meta = meta.loc[genera_clr.index]

# 2. Extract and Clean Metadata (Age & Gender)
meta_subset = meta[["consent_age", "Gender"]].copy()
meta_subset["consent_age"] = meta_subset["consent_age"].fillna(meta_subset["consent_age"].median())
meta_subset["Gender"] = meta_subset["Gender"].map({"Male": 0, "Female": 1})
meta_subset["Gender"] = meta_subset["Gender"].fillna(meta_subset["Gender"].mode()[0])

# Combine everything
X_all = pd.concat([genera_clr, meta_subset], axis=1, join='inner')

# Create binary datasets
def make_binary_dataset(meta, features, group_a, group_b):
    mask = meta["Study.Group"].isin([group_a, group_b])
    meta_sub = meta[mask]
    X = features.loc[meta_sub.index]
    y = (meta_sub["Study.Group"] == group_a).astype(int)
    return X, y

X_uc, y_uc = make_binary_dataset(meta, X_all, "UC", "nonIBD")
X_cd, y_cd = make_binary_dataset(meta, X_all, "CD", "nonIBD")

# Define column groups for the ColumnTransformer
microbe_cols = genera_clr.columns.tolist()
clinical_cols = ["consent_age", "Gender"]

print(f"Data Prep Complete.")
print(f"UC vs nonIBD: {X_uc.shape[0]} samples | CD vs nonIBD: {X_cd.shape[0]} samples")

Data Prep Complete.
UC vs nonIBD: 56 samples | CD vs nonIBD: 75 samples


In [12]:
# Custom Kruskal-Wallis Selector (Required for Pipeline)
class KruskalSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k=10):
        self.k = k
        self.selected_features_ = None
        
    def fit(self, X, y):
        p_values = []
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        y_arr = y.values if isinstance(y, pd.Series) else y
        
        for i in range(X_arr.shape[1]):
            group0 = X_arr[:, i][y_arr == 0]
            group1 = X_arr[:, i][y_arr == 1]
            stat, p = kruskal(group0, group1)
            p_values.append(p)
            
        k_actual = min(self.k, X_arr.shape[1])
        self.selected_indices_ = np.argsort(p_values)[:k_actual]
        return self
        
    def transform(self, X):
        X_arr = X.values if isinstance(X, pd.DataFrame) else X
        return X_arr[:, self.selected_indices_]

# Robust 3-Run Evaluation Function (FIXED)
def evaluate_multiple_runs(pipeline, X, y, task_name, n_runs=3, n_splits=5):
    metrics = {'roc_auc': [], 'accuracy': [], 'f1': []}
    
    for run in range(n_runs):
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42 + run)
        scores = cross_validate(pipeline, X, y, cv=cv, scoring=['roc_auc', 'accuracy', 'f1'], n_jobs=-1)
        for metric in metrics.keys():
            metrics[metric].extend(scores[f'test_{metric}'])
            
    summary = {}
    for metric, values in metrics.items():
        # Safely assign names to avoid capitalization bugs
        if metric == 'roc_auc':
            clean_name = "AUC"
        elif metric == 'f1':
            clean_name = "F1-Score"
        else:
            clean_name = metric.capitalize()
            
        summary[clean_name] = f"{np.mean(values):.3f} ± {np.std(values):.3f}"
        
    return pd.DataFrame([summary], index=[task_name])

### Integrating Metadata into UC vs nonIBD (Kruskal-Wallis + SVM)
We utilize a `ColumnTransformer` to route the 2,761 microbes through the `KruskalSelector` (k=250) and `StandardScaler`, while simultaneously routing Age and Gender purely through a `StandardScaler`. The concatenated 252 features are then passed to the RBF SVM.

In [13]:
print("=== Ultra-Optimizing UC vs nonIBD (Microbes + Age + Gender) ===")

# 1. The UC Traffic Cop (Leave k empty, GridSearch will fill it)
uc_preprocessor = ColumnTransformer(
    transformers=[
        ('microbes', Pipeline([
            ('kw', KruskalSelector()), 
            ('scaler', StandardScaler())
        ]), microbe_cols),
        ('clinical', StandardScaler(), clinical_cols)
    ]
)

uc_combined_base = Pipeline([
    ('preprocessor', uc_preprocessor),
    ('svm', SVC(probability=True, class_weight='balanced', random_state=42))
])

# 2. Your exact UC Ultra Grid, mapped to the new nested pipeline structure
c_space_uc = np.logspace(-3, 3, 20)
gamma_space_uc = list(np.logspace(-4, 1, 10)) + ['scale', 'auto']

uc_param_grid = {
    'preprocessor__microbes__kw__k': [100, 110, 120, 125, 130, 140, 150, 160, 175, 200, 250], 
    'svm__C': c_space_uc,
    'svm__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'svm__gamma': gamma_space_uc,
    'svm__degree': [2, 3, 4],
    'svm__coef0': [0.0, 0.1, 0.5, 1.0]
}

# 3. Search!
uc_search = RandomizedSearchCV(
    uc_combined_base, 
    param_distributions=uc_param_grid, 
    n_iter=500, 
    scoring='roc_auc', 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
    n_jobs=-1, 
    random_state=42,
    verbose=1
)

uc_search.fit(X_uc, y_uc)
best_uc_model = uc_search.best_estimator_

print(f"\nBest UC Combined Grid AUC: {uc_search.best_score_:.4f}")
print(f"Best Parameters: {uc_search.best_params_}\n")

# 4. Evaluate using the robust 3-run method
df_uc_combined = evaluate_multiple_runs(best_uc_model, X_uc, y_uc, "UC (Microbes + Metadata)")
display(df_uc_combined)

=== Ultra-Optimizing UC vs nonIBD (Microbes + Age + Gender) ===
Fitting 5 folds for each of 500 candidates, totalling 2500 fits

Best UC Combined Grid AUC: 0.6600
Best Parameters: {'svm__kernel': 'rbf', 'svm__gamma': np.float64(0.05994842503189409), 'svm__degree': 3, 'svm__coef0': 1.0, 'svm__C': np.float64(0.6951927961775606), 'preprocessor__microbes__kw__k': 250}



,AUC,Accuracy,F1-Score
UC (Microbes + Metadata),0.635 ± 0.150,0.536 ± 0.018,0.565 ± 0.282


### Integrating Metadata into CD vs nonIBD (PCA + SVM)
Here, the `ColumnTransformer` routes the microbes through a `StandardScaler` and `PCA` (n=22), while Age and Gender are scaled and appended. The final 24-feature vector is classified by our optimized Sigmoid SVM.

In [14]:
print("=== Ultra-Optimizing CD vs nonIBD (Microbes + Age + Gender) ===")

# 1. The CD Traffic Cop (Leave n_components empty, GridSearch will fill it)
cd_preprocessor = ColumnTransformer(
    transformers=[
        ('microbes', Pipeline([
            ('scaler', StandardScaler()),
            ('pca', PCA(random_state=42)) 
        ]), microbe_cols),
        ('clinical', StandardScaler(), clinical_cols)
    ]
)

cd_combined_base = Pipeline([
    ('preprocessor', cd_preprocessor),
    ('svm', SVC(probability=True, class_weight='balanced', random_state=42))
])

# 2. Your exact CD Ultra Grid, mapped to the new nested pipeline structure
c_space_cd = np.logspace(-2, 4, 25) 
gamma_space_cd = list(np.logspace(-5, -1, 15)) + ['scale', 'auto']

cd_param_grid = {
    'preprocessor__microbes__pca__n_components': [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 30, 35, 40], 
    'svm__C': c_space_cd,
    'svm__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'svm__gamma': gamma_space_cd,
    'svm__degree': [2, 3, 4],
    'svm__coef0': [0.0, 0.1, 0.5, 1.0, 2.0] 
}

# 3. Search!
cd_search = RandomizedSearchCV(
    cd_combined_base, 
    param_distributions=cd_param_grid, 
    n_iter=500, 
    scoring='roc_auc', 
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 
    n_jobs=-1, 
    random_state=42,
    verbose=1
)

cd_search.fit(X_cd, y_cd)
best_cd_model = cd_search.best_estimator_

print(f"\nBest CD Combined Grid AUC: {cd_search.best_score_:.4f}")
print(f"Best Parameters: {cd_search.best_params_}\n")

# 4. Evaluate using the robust 3-run method
df_cd_combined = evaluate_multiple_runs(best_cd_model, X_cd, y_cd, "CD (Microbes + Metadata)")
display(df_cd_combined)

=== Ultra-Optimizing CD vs nonIBD (Microbes + Age + Gender) ===
Fitting 5 folds for each of 500 candidates, totalling 2500 fits

Best CD Combined Grid AUC: 0.7289
Best Parameters: {'svm__kernel': 'sigmoid', 'svm__gamma': np.float64(7.196856730011514e-05), 'svm__degree': 2, 'svm__coef0': 1.0, 'svm__C': np.float64(5.623413251903491), 'preprocessor__microbes__pca__n_components': 22}



,AUC,Accuracy,F1-Score
CD (Microbes + Metadata),0.679 ± 0.145,0.591 ± 0.142,0.642 ± 0.166


In [15]:
# ==========================================
# FINAL CONCLUSION TABLE
# ==========================================
print("\n=== FINAL IMPACT OF METADATA ===")
# Note: These are your baselines from the end of notebook 05b
baseline_uc_auc = 0.647 
baseline_cd_auc = 0.679

new_uc_auc = float(df_uc_combined["AUC"].iloc[0].split(" ")[0])
new_cd_auc = float(df_cd_combined["AUC"].iloc[0].split(" ")[0])

print(f"UC AUC Change: {baseline_uc_auc:.3f} -> {new_uc_auc:.3f} (Δ {new_uc_auc - baseline_uc_auc:+.3f})")
print(f"CD AUC Change: {baseline_cd_auc:.3f} -> {new_cd_auc:.3f} (Δ {new_cd_auc - baseline_cd_auc:+.3f})")


=== FINAL IMPACT OF METADATA ===
UC AUC Change: 0.647 -> 0.635 (Δ -0.012)
CD AUC Change: 0.679 -> 0.679 (Δ +0.000)


In [16]:
print("=== Saving Final Multi-Modal Models ===")

# Save the UC combined model
joblib.dump(best_uc_model, '../results/final_model_uc_combined.pkl')
print("✅ Saved UC Combined Model to: '../results/final_model_uc_combined.pkl'")

# Save the CD combined model
joblib.dump(best_cd_model, '../results/final_model_cd_combined.pkl')
print("✅ Saved CD Combined Model to: '../results/final_model_cd_combined.pkl'")

print("\n🎉 Project pipeline successfully completed and saved!")

=== Saving Final Multi-Modal Models ===
✅ Saved UC Combined Model to: '../results/final_model_uc_combined.pkl'
✅ Saved CD Combined Model to: '../results/final_model_cd_combined.pkl'

🎉 Project pipeline successfully completed and saved!
